# AI Shield: Forensic + Semantic Adaptation v1
**Research only. The frozen CF detector, app and original checkpoints are never modified.**

Question: how much can the two main networks improve after limited adaptation to Dataset v1?
This is an **8-feature two-branch variant**, not a retraining of the original 17-feature pipeline.
It uses EfficientNet-B0 and the original custom ViT attention head. VAE and face branches are excluded.
The old full pipeline and original/fine-tuned CF are historical comparison rows, not fusion inputs.

1. Check existing data/checkpoints and reproduce fixed audit predictions.
2. Fit new fusion with both original networks frozen.
3. Adapt each network for 3 head epochs + 3 limited-backbone epochs; refit fusion.
4. Lock validation-selected checkpoints/thresholds, then evaluate the historical tests.
5. Measure batch-1 latency against frozen CF on the same runtime and images.

Of the original **2,990 training images**, about 80% train the branches and 20% train fusion.
The exact split is by connected prompt/source/duplicate groups, so counts vary.
Validation stays at 640. CF used all 2,990 for its own branch training: this comparison matches
the available data budget, **not identical per-network exposure or compute**. Original pretraining costs are unknown.
The already-inspected test and GPT Image 2 results are historical comparisons, not a fresh blind confirmation.

Select a Colab GPU runtime, then run the cells in order. Mounting your Drive requires your approval.
A real GPU run has not been performed just by generating this notebook. Failed checks must be investigated, not bypassed.


In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'timm==1.0.15', 'transformers==4.57.6', 'scikit-learn==1.6.1', 'pandas==2.2.3', 'numpy==2.1.3', 'Pillow==11.3.0', 'psutil>=5.9,<8', 'tqdm>=4.66,<5', 'joblib>=1.3,<2'])


## 1. Load the embedded, versioned research code
No GitHub push, model upload or API changes are needed.


In [ ]:
EMBEDDED_SOURCES = {'src/ai_shield_adaptation.py': '"""Controlled two-branch adaptation experiment. Never imported by the app."""\nfrom __future__ import annotations\n\nfrom dataclasses import asdict, dataclass\nfrom datetime import datetime, timezone\nimport hashlib\nimport importlib.metadata\nimport json\nfrom pathlib import Path\nimport random\nimport time\nimport uuid\n\nimport numpy as np\nimport pandas as pd\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.metrics import precision_recall_curve\nfrom sklearn.model_selection import GroupShuffleSplit\nfrom sklearn.pipeline import make_pipeline\nfrom sklearn.preprocessing import StandardScaler\n\nfrom src.ai_shield_metrics import binary_metrics\n\n\nEXPECTED_COUNTS = {"train": 2990, "validation": 640, "test": 640, "unseen_generator": 854}\nSPLIT_HASHES = {\n    "train": "29e893703856c3e1c5e71a0a5f04619ef01bb0e7cc8829a01e39ce605fc9f2a2",\n    "validation": "1d69e4c55ab1896b51421b2e9bbc3d1a919966087ca7601f8f3f3919250aaf11",\n    "test": "fa53c8831a4d4657d1ea34558b284d13dc65935686255761f89638c6802af063",\n    "unseen_generator": "3fc0d4ea61bab0340b9512e732042bddb3f066768aadb9c1dbc574c05fb8d392",\n}\nSEMANTIC_ID = "google/vit-base-patch16-224-in21k"\nSEMANTIC_REVISION = "b4569560a39a0f1af58e3ddaf17facf20ab919b0"\nFEATURES = [\n    "effnet_logit", "effnet_prob", "semantic_vit_logit", "semantic_vit_prob",\n    "semantic_attention_entropy", "semantic_attention_focus",\n    "semantic_self_attention_entropy", "semantic_self_attention_focus",\n]\n\n\n@dataclass(frozen=True)\nclass Config:\n    drive_root: str = "/content/drive/MyDrive"\n    seed: int = 191\n    fusion_fraction: float = 0.20\n    head_epochs: int = 3\n    partial_epochs: int = 3\n    head_lr: float = 3e-4\n    partial_head_lr: float = 1e-4\n    backbone_lr: float = 1e-5\n    weight_decay: float = 1e-2\n    batch_size: int = 16\n    num_workers: int = 0\n\n    @property\n    def root(self):\n        return Path(self.drive_root)\n\n    @property\n    def split_root(self):\n        return self.root / "AI_Shield_Data/splits/v1"\n\n    @property\n    def cf_root(self):\n        return self.root / "AI_Shield_CommunityForensics/experiments/finetune_v1"\n\n    @property\n    def audit_root(self):\n        return self.root / "AI_Shield_Baseline_Evaluation/saved_pipeline_v1/results"\n\n    @property\n    def checkpoints(self):\n        return {\n            "effnet": self.root / "AI_Shield_Phase1/effnet_b0_tinygenimage_full_balanced_v1/effnet_b0_tinygenimage_full_balanced_v1_best.pt",\n            "semantic": self.root / "AI_Shield_Phase2_Semantic/semantic_vit_transformer_attention_v1/checkpoints/semantic_vit_transformer_attention_best.pt",\n        }\n\n\ndef sha256_file(path):\n    digest = hashlib.sha256()\n    with Path(path).open("rb") as handle:\n        for chunk in iter(lambda: handle.read(1024 * 1024), b""):\n            digest.update(chunk)\n    return digest.hexdigest()\n\n\ndef write_json(path, payload):\n    # Exclusive creation makes a repeated cell fail instead of replacing evidence.\n    with Path(path).open("x", encoding="utf-8") as handle:\n        json.dump(payload, handle, indent=2, allow_nan=False)\n        handle.write("\\n")\n\n\ndef checked_csv(path):\n    return pd.read_csv(path, keep_default_na=False, dtype={\n        "sample_id": str, "sha256": str, "group_id": str, "prompt_id": str,\n    })\n\n\ndef validate_frames(frames, expected_counts=None):\n    if set(frames) != set(EXPECTED_COUNTS):\n        raise ValueError("Expected train, validation, test and unseen_generator splits.")\n    required = {"sample_id", "label", "split", "sha256", "group_id", "prompt_id", "generator", "local_path"}\n    for name, frame in frames.items():\n        if not required.issubset(frame):\n            raise ValueError(f"{name}: missing columns {required - set(frame)}")\n        if expected_counts and len(frame) != expected_counts[name]:\n            raise ValueError(f"{name}: wrong sample count")\n        if set(frame.label) != {0, 1} or set(frame.split) != {name}:\n            raise ValueError(f"{name}: invalid labels or split values")\n        for column in ("sample_id", "group_id", "sha256", "local_path"):\n            if frame[column].isna().any() or frame[column].astype(str).str.strip().eq("").any():\n                raise ValueError(f"{name}: empty {column}")\n        if not frame.sha256.str.fullmatch(r"[a-f0-9]{64}").all():\n            raise ValueError(f"{name}: malformed SHA-256")\n    all_rows = pd.concat(frames.values(), ignore_index=True)\n    if all_rows.sample_id.duplicated().any():\n        raise ValueError("Duplicate sample IDs")\n    for column in ("sha256", "group_id", "local_path"):\n        if (all_rows.groupby(column).split.nunique() > 1).any():\n            raise ValueError(f"{column} leakage across outer splits")\n    if (all_rows.groupby("sha256").label.nunique() > 1).any():\n        raise ValueError("Identical image bytes have conflicting labels")\n    dev = all_rows[all_rows.split != "unseen_generator"]\n    fake = dev[dev.label == 1]\n    if fake.generator.eq("gpt-image-2").any():\n        raise ValueError("Held-out generator entered development")\n    prompts = fake[fake.prompt_id.astype(str).str.strip().ne("")]\n    if (prompts.groupby("prompt_id").split.nunique() > 1).any():\n        raise ValueError("Prompt leakage across development splits")\n    unseen = frames["unseen_generator"]\n    if set(unseen.loc[unseen.label == 1, "generator"]) != {"gpt-image-2"}:\n        raise ValueError("Unexpected held-out generator")\n\n\ndef partition_training(train, fraction=0.20, seed=191):\n    if set(train.split) != {"train"}:\n        raise ValueError("Inner partition must only use the original training split")\n    if not 0 < fraction < 1:\n        raise ValueError("Fusion fraction must be in (0, 1)")\n    train = train.reset_index(drop=True)\n    # Connected groups keep prompts, source groups and duplicate bytes together.\n    parent = list(range(len(train)))\n\n    def find(index):\n        while parent[index] != index:\n            parent[index] = parent[parent[index]]\n            index = parent[index]\n        return index\n\n    for column in ("group_id", "sha256", "prompt_id"):\n        seen = {}\n        for index, row in train.iterrows():\n            value = str(row[column]).strip()\n            if not value or (column == "prompt_id" and row.label != 1):\n                continue\n            if value in seen:\n                parent[find(index)] = find(seen[value])\n            else:\n                seen[value] = index\n    groups = [find(i) for i in range(len(train))]\n    splitter = GroupShuffleSplit(n_splits=1, test_size=fraction, random_state=seed)\n    branch, fusion = next(splitter.split(train, train.label, groups))\n    result = train.copy()\n    result["role"] = "branch_train"\n    result.loc[fusion, "role"] = "fusion_train"\n    for role, subset in result.groupby("role"):\n        if set(subset.label) != {0, 1}:\n            raise ValueError(f"{role} needs both classes; do not silently resplit")\n    return result\n\n\ndef align_predictions(reference, predictions, columns):\n    required = {"sample_id", "label", *columns}\n    if not required.issubset(predictions):\n        raise ValueError(f"Missing prediction columns: {required - set(predictions)}")\n    if reference.sample_id.duplicated().any() or predictions.sample_id.duplicated().any():\n        raise ValueError("Duplicate prediction IDs")\n    if set(reference.sample_id) != set(predictions.sample_id):\n        raise ValueError("Prediction IDs differ from the manifest; no partial comparisons")\n    ordered = predictions.set_index("sample_id").loc[reference.sample_id].reset_index()\n    if not np.array_equal(reference.label.to_numpy(), ordered.label.to_numpy()):\n        raise ValueError("Prediction labels do not match manifest")\n    if not np.isfinite(ordered[columns].to_numpy(dtype=float)).all():\n        raise ValueError("Non-finite predictions")\n    return ordered\n\n\ndef max_f1_threshold(labels, scores):\n    labels, scores = np.asarray(labels), np.asarray(scores, dtype=float)\n    if set(labels) != {0, 1} or not np.isfinite(scores).all() or ((scores < 0) | (scores > 1)).any():\n        raise ValueError("Need both classes and finite [0, 1] scores")\n    precision, recall, thresholds = precision_recall_curve(labels, scores)\n    f1 = 2 * precision[:-1] * recall[:-1] / np.maximum(precision[:-1] + recall[:-1], 1e-15)\n    return float(thresholds[np.flatnonzero(f1 == f1.max())[-1]])\n\n\ndef fit_fusion(predictions, branch_training_ids):\n    if set(predictions.split) != {"train"}:\n        raise ValueError("Fusion fitting is restricted to training images")\n    if set(predictions.sample_id) & set(branch_training_ids):\n        raise ValueError("Fusion images were used to fine-tune a branch")\n    align_predictions(predictions, predictions, FEATURES)\n    if set(predictions.label) != {0, 1}:\n        raise ValueError("Fusion needs both classes")\n    model = make_pipeline(StandardScaler(), LogisticRegression(C=1.0, max_iter=2000, random_state=191))\n    model.fit(predictions[FEATURES], predictions.label)\n    return model\n\n\ndef preflight(config):\n    frames = {}\n    for name in EXPECTED_COUNTS:\n        path = config.split_root / f"{name}.csv"\n        if sha256_file(path) != SPLIT_HASHES[name]:\n            raise ValueError(f"{name}: differs from the recorded CF split")\n        frames[name] = checked_csv(path)\n    validate_frames(frames, EXPECTED_COUNTS)\n    audit = json.loads((config.audit_root / "saved_pipeline_evaluation_report.json").read_text())\n    for branch, path in config.checkpoints.items():\n        if sha256_file(path) != audit["source_artifacts"][f"{branch}_sha256"]:\n            raise ValueError(f"{branch}: checkpoint differs from the saved baseline audit")\n    from tqdm.auto import tqdm\n    for row in tqdm(pd.concat(frames.values()).itertuples(), total=sum(EXPECTED_COUNTS.values()), desc="Verify image bytes"):\n        if sha256_file(row.local_path) != row.sha256:\n            raise ValueError(f"Changed image: {row.sample_id}")\n    return frames, audit\n\n\ndef prepare_run(config, frames):\n    if config.head_epochs < 1 or config.partial_epochs < 0 or config.batch_size < 1:\n        raise ValueError("Invalid training budget")\n    partition = partition_training(frames["train"], config.fusion_fraction, config.seed)\n    run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid.uuid4().hex[:8]\n    run = config.root / "AI_Shield_Adaptation" / "two_branch_v1" / run_id\n    run.mkdir(parents=True, exist_ok=False)\n    partition.to_csv(run / "training_roles.csv", index=False)\n    versions = {name: importlib.metadata.version(name) for name in (\n        "torch", "torchvision", "timm", "transformers", "scikit-learn", "pandas", "numpy", "Pillow",\n    )}\n    write_json(run / "experiment_plan.json", {\n        "config": asdict(config), "libraries": versions, "split_sha256": SPLIT_HASHES,\n        "source_sha256": {k: sha256_file(v) for k, v in config.checkpoints.items()},\n        "implementation_sha256": sha256_file(__file__),\n        "semantic_revision": SEMANTIC_REVISION, "feature_order": FEATURES,\n        "training_roles_sha256": sha256_file(run / "training_roles.csv"),\n        "counts_by_role": partition.role.value_counts().to_dict(),\n        "scope": "Two-branch variant, NOT the original 17-feature full pipeline. No VAE/face branch.",\n        "budget_note": "Same 2990-image development budget as CF; branch training uses only the branch_train subset. CF trained on all 2990.",\n        "preprocessing": "Original audit: frame 0, RGB without EXIF transpose; EffNet resize256 bilinear/ImageNet; semantic pinned HF processor. Train-only horizontal flip 0.5.",\n        "test_policy": "Previously examined historical tests; never used for fitting/selection. Fresh confirmation still needed.",\n        "pretraining_cost": "Not measured; training times cover only this adaptation, not original pretraining.",\n    })\n    return run, partition\n\n\ndef seed_everything(seed):\n    import torch\n    random.seed(seed)\n    np.random.seed(seed)\n    torch.manual_seed(seed)\n    if torch.cuda.is_available():\n        torch.cuda.manual_seed_all(seed)\n        torch.backends.cuda.matmul.allow_tf32 = False\n        torch.backends.cudnn.allow_tf32 = False\n        torch.backends.cudnn.benchmark = False\n\n\ndef open_rgb(path):\n    from PIL import Image\n    with Image.open(path) as image:\n        image.seek(0)\n        return image.convert("RGB").copy()\n\n\ndef make_loader(frame, branch, processor, config, training=False):\n    import torch\n    from torch.utils.data import DataLoader, Dataset\n    from torchvision import transforms\n    transform = transforms.Compose([\n        transforms.Resize((256, 256), interpolation=transforms.InterpolationMode.BILINEAR, antialias=True),\n        transforms.ToTensor(), transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),\n    ])\n\n    class Images(Dataset):\n        def __len__(self):\n            return len(frame)\n\n        def __getitem__(self, index):\n            from PIL import Image\n            row = frame.iloc[index]\n            image = open_rgb(row.local_path)\n            if training and random.random() < 0.5:\n                image = image.transpose(Image.Transpose.FLIP_LEFT_RIGHT)\n            x = transform(image) if branch == "effnet" else processor(images=image, return_tensors="pt")["pixel_values"][0]\n            return x, float(row.label), str(row.sample_id)\n\n    # Zero workers is deliberate: the notebook-local dataset also works on Windows.\n    if config.num_workers != 0:\n        raise ValueError("This portable loader uses num_workers=0")\n    generator = torch.Generator().manual_seed(config.seed)\n    return DataLoader(Images(), batch_size=config.batch_size, shuffle=training, num_workers=0, generator=generator)\n\n\ndef predict_branch(model, loader, frame, branch, device):\n    import torch\n    model.to(device).eval()\n    rows = []\n    with torch.inference_mode():\n        for x, labels, ids in loader:\n            result = model(x.to(device))\n            logits = result if branch == "effnet" else result["logits"]\n            if not torch.isfinite(logits).all():\n                raise ValueError("Non-finite model logits")\n            scores = torch.sigmoid(logits).cpu().tolist()\n            for i, sample_id in enumerate(ids):\n                prefix = "effnet" if branch == "effnet" else "semantic_vit"\n                row = {"sample_id": sample_id, "label": int(labels[i]),\n                       f"{prefix}_logit": float(logits[i].cpu()), f"{prefix}_prob": scores[i]}\n                if branch == "semantic":\n                    for key in ("attention_entropy", "attention_focus", "self_attention_entropy", "self_attention_focus"):\n                        row[f"semantic_{key}"] = float(result[key][i].cpu())\n                rows.append(row)\n    columns = FEATURES[:2] if branch == "effnet" else FEATURES[2:]\n    result = align_predictions(frame, pd.DataFrame(rows), columns)\n    result["split"] = frame.split.to_numpy()\n    return result\n\n\ndef predict_pair(models, processor, frame, config, device):\n    results = []\n    for branch, model in models.items():\n        loader = make_loader(frame, branch, processor, config)\n        results.append(predict_branch(model, loader, frame, branch, device))\n        model.cpu()\n    merged = results[0].merge(results[1], on=["sample_id", "label", "split"], validate="one_to_one")\n    return align_predictions(frame, merged, FEATURES)\n\n\ndef check_audit_parity(models, processor, frames, config, device):\n    # This is a fixed implementation check, not a metric used for model selection.\n    fixture = frames["test"].sort_values("sample_id").groupby("label", group_keys=False).head(3)\n    actual = predict_pair(models, processor, fixture, config, device)\n    saved = checked_csv(config.audit_root / "aishield_test_predictions.csv")\n    saved = saved[saved.sample_id.isin(fixture.sample_id)]\n    saved = align_predictions(fixture, saved, FEATURES)\n    differences = {}\n    for column in FEATURES:\n        a, b = actual[column].to_numpy(), saved[column].to_numpy()\n        differences[column] = float(np.max(np.abs(a - b)))\n        if not np.allclose(a, b, atol=5e-4, rtol=1e-4):\n            raise ValueError(f"Saved baseline parity failed: {column}. Stop before training; investigate preprocessing/backbone versions.")\n    return {"n": len(fixture), "sample_ids": fixture.sample_id.tolist(), "max_absolute_errors": differences}\n\n\ndef configure_training(model, branch, stage):\n    model.requires_grad_(False)\n    if branch == "effnet":\n        head = model.model.classifier\n        backbone = [model.model.blocks[-1], model.model.conv_head, model.model.bn2]\n    else:\n        head = model.semantic_head\n        backbone = [model.vit.encoder.layer[-1], model.vit.layernorm]\n    head.requires_grad_(True)\n    if stage == "partial":\n        for module in backbone:\n            module.requires_grad_(True)\n    elif stage != "head":\n        raise ValueError("Unknown training stage")\n    return head, backbone\n\n\ndef training_mode(model, branch, stage, head, backbone):\n    # requires_grad=False alone would not freeze BatchNorm running statistics.\n    model.eval()\n    head.train()\n    if branch == "effnet":\n        model.model.training = True  # classifier\'s functional dropout only\n    if stage == "partial":\n        for module in backbone:\n            module.train()\n\n\ndef train_branch(model, branch, processor, branch_frame, validation, config, run, device):\n    import torch\n    from sklearn.metrics import roc_auc_score\n    if set(branch_frame.split) != {"train"} or set(validation.split) != {"validation"}:\n        raise ValueError("Wrong training/selection split")\n    seed_everything(config.seed)\n    checkpoint = Path(run) / f"{branch}_adapted.pt"\n    if checkpoint.exists():\n        raise FileExistsError("Start a new run instead of overwriting a trained branch")\n    model.to(device)\n    train_loader = make_loader(branch_frame, branch, processor, config, training=True)\n    val_loader = make_loader(validation, branch, processor, config)\n    score_column = "effnet_prob" if branch == "effnet" else "semantic_vit_prob"\n    initial = predict_branch(model, val_loader, validation, branch, device)\n    best_auc = float(roc_auc_score(initial.label, initial[score_column]))\n    selected = {"stage": "original", "epoch": 0, "validation_auc": best_auc}\n    torch.save({"model_state": model.state_dict(), **selected}, checkpoint)\n    history, max_trainable = [], 0\n    if str(device).startswith("cuda"):\n        torch.cuda.synchronize()\n        torch.cuda.reset_peak_memory_stats()\n    start = time.perf_counter()\n    for stage, epochs in (("head", config.head_epochs), ("partial", config.partial_epochs)):\n        if epochs == 0:\n            continue\n        model.load_state_dict(torch.load(checkpoint, map_location=device, weights_only=True)["model_state"], strict=True)\n        head, backbone = configure_training(model, branch, stage)\n        head_ids = {id(p) for p in head.parameters()}\n        groups = [{"params": list(head.parameters()), "lr": config.head_lr if stage == "head" else config.partial_head_lr}]\n        others = [p for p in model.parameters() if p.requires_grad and id(p) not in head_ids]\n        if others:\n            groups.append({"params": others, "lr": config.backbone_lr})\n        optimizer = torch.optim.AdamW(groups, weight_decay=config.weight_decay)\n        trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)\n        max_trainable = max(max_trainable, trainable)\n        for epoch in range(1, epochs + 1):\n            training_mode(model, branch, stage, head, backbone)\n            loss_sum, count = 0.0, 0\n            epoch_start = time.perf_counter()\n            for x, labels, _ in train_loader:\n                optimizer.zero_grad(set_to_none=True)\n                output = model(x.to(device))\n                logits = output if branch == "effnet" else output["logits"]\n                loss = torch.nn.functional.binary_cross_entropy_with_logits(logits, labels.to(device, dtype=torch.float32))\n                if not torch.isfinite(loss):\n                    raise ValueError("Non-finite training loss")\n                loss.backward()\n                torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], 1.0, error_if_nonfinite=True)\n                optimizer.step()\n                loss_sum += float(loss.detach()) * len(labels)\n                count += len(labels)\n            predictions = predict_branch(model, val_loader, validation, branch, device)\n            auc = float(roc_auc_score(predictions.label, predictions[score_column]))\n            row = {"branch": branch, "stage": stage, "epoch": epoch, "train_loss": loss_sum / count,\n                   "validation_auc": auc, "seconds_including_validation": time.perf_counter() - epoch_start,\n                   "trainable_parameters": trainable}\n            history.append(row)\n            print(row)\n            pd.DataFrame(history).to_csv(Path(run) / f"{branch}_history.csv", index=False)\n            if auc > best_auc:\n                best_auc = auc\n                selected = {"stage": stage, "epoch": epoch, "validation_auc": auc}\n                torch.save({"model_state": model.state_dict(), **selected}, checkpoint)\n    if str(device).startswith("cuda"):\n        torch.cuda.synchronize()\n    elapsed = time.perf_counter() - start\n    peak = torch.cuda.max_memory_allocated() if str(device).startswith("cuda") else None\n    model.load_state_dict(torch.load(checkpoint, map_location=device, weights_only=True)["model_state"], strict=True)\n    model.cpu().eval()\n    report = {"branch": branch, "selected": selected, "training_images": len(branch_frame),\n              "adaptation_seconds_including_validation_and_saves": elapsed,\n              "max_trainable_parameters": max_trainable, "total_parameters": sum(p.numel() for p in model.parameters()),\n              "cuda_peak_allocated_bytes": peak, "checkpoint_sha256": sha256_file(checkpoint)}\n    write_json(Path(run) / f"{branch}_training.json", report)\n    return report\n\n\ndef fit_variant(name, models, processor, partition, validation, config, run, device):\n    import joblib\n    fusion_rows = partition[partition.role == "fusion_train"]\n    branch_ids = partition.loc[partition.role == "branch_train", "sample_id"]\n    train_predictions = predict_pair(models, processor, fusion_rows, config, device)\n    validation_predictions = predict_pair(models, processor, validation, config, device)\n    start = time.perf_counter()\n    fusion = fit_fusion(train_predictions, branch_ids)\n    fit_seconds = time.perf_counter() - start\n    validation_predictions["fusion_prob"] = fusion.predict_proba(validation_predictions[FEATURES])[:, 1]\n    thresholds = {key: max_f1_threshold(validation_predictions.label, validation_predictions[key])\n                  for key in ("effnet_prob", "semantic_vit_prob", "fusion_prob")}\n    write_json(Path(run) / f"{name}_selection.json", {\n        "thresholds": thresholds, "selection_split": "validation", "fusion_fit_seconds": fit_seconds,\n        "fusion_training_images": len(fusion_rows), "feature_order": FEATURES,\n    })\n    joblib.dump(fusion, Path(run) / f"{name}_fusion.joblib")\n    train_predictions.to_csv(Path(run) / f"{name}_fusion_training_predictions.csv", index=False)\n    validation_predictions.to_csv(Path(run) / f"{name}_validation_predictions.csv", index=False)\n    return fusion, thresholds\n\n\ndef scored_metrics(labels, scores, threshold):\n    if not np.isfinite(scores).all() or ((np.asarray(scores) < 0) | (np.asarray(scores) > 1)).any():\n        raise ValueError("Invalid detection scores")\n    result = binary_metrics(labels, scores, threshold)\n    result["fpr"] = result["fp"] / (result["fp"] + result["tn"])\n    return result\n\n\ndef evaluate_variant(name, models, processor, frames, fusion, thresholds, config, run, device):\n    rows = []\n    for split in ("test", "unseen_generator"):\n        predictions = predict_pair(models, processor, frames[split], config, device)\n        predictions["fusion_prob"] = fusion.predict_proba(predictions[FEATURES])[:, 1]\n        predictions.to_csv(Path(run) / f"{name}_{split}_predictions.csv", index=False)\n        for column in ("effnet_prob", "semantic_vit_prob", "fusion_prob"):\n            rows.append({"model": f"{name}/{column}", "split": split,\n                         **scored_metrics(predictions.label, predictions[column], thresholds[column])})\n    return rows\n\n\ndef historical_references(frames, config, audit, run):\n    rows, evidence = [], {}\n    threshold_path = config.cf_root / "results/validation_thresholds.json"\n    thresholds = json.loads(threshold_path.read_text())\n    evidence[str(threshold_path)] = sha256_file(threshold_path)\n    old_metrics = [r for r in audit["metrics"] if "Full fusion" in r["model"]]\n    for split in ("test", "unseen_generator"):\n        for key in ("pretrained", "finetuned"):\n            path = config.cf_root / f"predictions/{key}_{split}.csv"\n            pred = align_predictions(frames[split], checked_csv(path), ["prob_fake"])\n            threshold = thresholds["baseline" if key == "pretrained" else key]["max_f1"]\n            rows.append({"model": f"CF/{key}", "split": split,\n                         **scored_metrics(pred.label, pred.prob_fake, threshold)})\n            evidence[str(path)] = sha256_file(path)\n        matches = [r for r in old_metrics if r["split"] == split]\n        if len(matches) != 1:\n            raise ValueError("Missing/ambiguous original full fusion reference")\n        path = config.audit_root / f"aishield_{split}_predictions.csv"\n        pred = align_predictions(frames[split], checked_csv(path), ["aishield_full_prob"])\n        rows.append({"model": "original_full_17_feature/reference_only", "split": split,\n                     **scored_metrics(pred.label, pred.aishield_full_prob, matches[0]["threshold"])})\n        evidence[str(path)] = sha256_file(path)\n    write_json(Path(run) / "historical_reference_hashes.json", evidence)\n    return rows\n\n\ndef benchmark(predict_one, paths, device, repeats=3, warmup=3):\n    """Warm batch-1 end-to-end latency; excludes checkpoint/model loading."""\n    import torch\n    import psutil\n    if not paths or repeats < 1:\n        raise ValueError("Need benchmark images and positive repeats")\n    def sync():\n        if str(device).startswith("cuda"):\n            torch.cuda.synchronize()\n    for path in paths[:warmup]:\n        predict_one(path)\n    sync()\n    if str(device).startswith("cuda"):\n        torch.cuda.reset_peak_memory_stats()\n    durations = []\n    for _ in range(repeats):\n        for path in paths:\n            sync()\n            start = time.perf_counter()\n            predict_one(path)\n            sync()\n            durations.append(time.perf_counter() - start)\n    return {"images": len(paths), "repeats": repeats, "batch_size": 1,\n            "median_seconds": float(np.median(durations)), "p95_seconds": float(np.quantile(durations, .95)),\n            "cuda_peak_allocated_bytes": torch.cuda.max_memory_allocated() if str(device).startswith("cuda") else None,\n            "process_rss_after_bytes": psutil.Process().memory_info().rss,\n            "memory_note": "RSS is the whole notebook process after this run, NOT isolated model peak memory.",\n            "scope": "warm file-open + preprocessing + transfer + model + score; no model load, API, provenance or network"}\n\n\nclass PairPredictor:\n    def __init__(self, models, processor, fusion, device):\n        from torchvision import transforms\n        self.models, self.processor, self.fusion, self.device = models, processor, fusion, device\n        self.transform = transforms.Compose([\n            transforms.Resize((256, 256), interpolation=transforms.InterpolationMode.BILINEAR, antialias=True),\n            transforms.ToTensor(), transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),\n        ])\n        for model in models.values():\n            model.to(device).eval()\n\n    def predict(self, path):\n        import torch\n        image = open_rgb(path)\n        with torch.inference_mode():\n            logit = self.models["effnet"](self.transform(image).unsqueeze(0).to(self.device))[0]\n            semantic = self.models["semantic"](self.processor(images=image, return_tensors="pt")["pixel_values"].to(self.device))\n            row = {"effnet_logit": float(logit), "effnet_prob": float(torch.sigmoid(logit)),\n                   "semantic_vit_logit": float(semantic["logits"][0]),\n                   "semantic_vit_prob": float(torch.sigmoid(semantic["logits"][0]))}\n            for key in ("attention_entropy", "attention_focus", "self_attention_entropy", "self_attention_focus"):\n                row["semantic_" + key] = float(semantic[key][0])\n        values = pd.DataFrame([row])[FEATURES]\n        if not np.isfinite(values.to_numpy()).all():\n            raise ValueError("Non-finite inference features")\n        return float(self.fusion.predict_proba(values)[0, 1])\n', 'src/ai_shield_adaptation_models.py': '"""Research models: original semantic head, audited loaders, strict checkpoints."""\nimport math\nfrom contextlib import nullcontext\nimport torch\nfrom torch import nn\n\nfrom src.ai_shield_adaptation import SEMANTIC_ID, SEMANTIC_REVISION\n\n\nclass CustomMultiHeadSelfAttention(nn.Module):\n    def __init__(self, d_model, num_heads, dropout=0.15):\n        super().__init__()\n        if d_model % num_heads != 0:\n            raise ValueError(f"d_model={d_model} must be divisible by num_heads={num_heads}")\n        self.d_model = d_model\n        self.num_heads = num_heads\n        self.head_dim = d_model // num_heads\n        self.scale = self.head_dim ** -0.5\n\n        self.q_proj = nn.Linear(d_model, d_model)\n        self.k_proj = nn.Linear(d_model, d_model)\n        self.v_proj = nn.Linear(d_model, d_model)\n        self.out_proj = nn.Linear(d_model, d_model)\n        self.dropout = nn.Dropout(dropout)\n\n    def _split_heads(self, x):\n        batch_size, num_tokens, _ = x.shape\n        x = x.view(batch_size, num_tokens, self.num_heads, self.head_dim)\n        return x.transpose(1, 2)\n\n    def _merge_heads(self, x):\n        batch_size, num_heads, num_tokens, head_dim = x.shape\n        x = x.transpose(1, 2).contiguous()\n        return x.view(batch_size, num_tokens, num_heads * head_dim)\n\n    def forward(self, x):\n        q = self._split_heads(self.q_proj(x))\n        k = self._split_heads(self.k_proj(x))\n        v = self._split_heads(self.v_proj(x))\n\n        scores = torch.matmul(q, k.transpose(-2, -1)) * self.scale\n        attention = torch.softmax(scores, dim=-1)\n        attention_for_summary = attention\n        attention = self.dropout(attention)\n\n        context = torch.matmul(attention, v)\n        context = self._merge_heads(context)\n        output = self.out_proj(context)\n        return output, attention_for_summary\n\n\nclass SemanticTransformerBlock(nn.Module):\n    def __init__(self, d_model, num_heads, mlp_ratio=2.0, dropout=0.15):\n        super().__init__()\n        hidden_dim = int(d_model * mlp_ratio)\n        self.norm1 = nn.LayerNorm(d_model)\n        self.attention = CustomMultiHeadSelfAttention(d_model, num_heads, dropout)\n        self.dropout1 = nn.Dropout(dropout)\n        self.norm2 = nn.LayerNorm(d_model)\n        self.mlp = nn.Sequential(\n            nn.Linear(d_model, hidden_dim),\n            nn.GELU(),\n            nn.Dropout(dropout),\n            nn.Linear(hidden_dim, d_model),\n            nn.Dropout(dropout),\n        )\n\n    def forward(self, x):\n        attn_input = self.norm1(x)\n        attn_output, attention = self.attention(attn_input)\n        x = x + self.dropout1(attn_output)\n        mlp_input = self.norm2(x)\n        x = x + self.mlp(mlp_input)\n        return x, attention\n\n\nclass SemanticTransformerHead(nn.Module):\n    def __init__(\n        self,\n        hidden_size,\n        d_model=256,\n        num_heads=4,\n        depth=1,\n        mlp_ratio=2.0,\n        attention_dim=256,\n        classifier_hidden_dim=256,\n        dropout=0.15,\n    ):\n        super().__init__()\n        self.input_projection = nn.Linear(hidden_size, d_model)\n        self.blocks = nn.ModuleList([\n            SemanticTransformerBlock(d_model, num_heads, mlp_ratio, dropout)\n            for _ in range(depth)\n        ])\n        self.final_norm = nn.LayerNorm(d_model)\n\n        self.pool_key = nn.Linear(d_model, attention_dim)\n        self.pool_energy = nn.Linear(attention_dim, 1)\n        self.classifier = nn.Sequential(\n            nn.Dropout(dropout),\n            nn.Linear(d_model, classifier_hidden_dim),\n            nn.GELU(),\n            nn.Dropout(dropout),\n            nn.Linear(classifier_hidden_dim, 1),\n        )\n\n    def _attention_summary(self, attention_maps):\n        eps = 1e-8\n        summaries = []\n        for attention in attention_maps:\n            num_tokens = attention.shape[-1]\n            entropy = -(attention * torch.log(attention + eps)).sum(dim=-1)\n            entropy = entropy / math.log(num_tokens)\n            focus = attention.max(dim=-1).values\n            summaries.append((entropy.mean(dim=(1, 2)), focus.mean(dim=(1, 2))))\n        entropy = torch.stack([item[0] for item in summaries], dim=0).mean(dim=0)\n        focus = torch.stack([item[1] for item in summaries], dim=0).mean(dim=0)\n        return entropy, focus\n\n    def forward(self, patch_tokens):\n        x = self.input_projection(patch_tokens)\n        self_attention_maps = []\n        for block in self.blocks:\n            x, attention = block(x)\n            self_attention_maps.append(attention)\n        x = self.final_norm(x)\n\n        pool_energy = self.pool_energy(torch.tanh(self.pool_key(x))).squeeze(-1)\n        pool_weights = torch.softmax(pool_energy, dim=1)\n        pooled = torch.sum(pool_weights.unsqueeze(-1) * x, dim=1)\n        logits = self.classifier(pooled).squeeze(-1)\n\n        eps = 1e-8\n        pool_entropy = -(pool_weights * torch.log(pool_weights + eps)).sum(dim=1)\n        pool_entropy = pool_entropy / math.log(pool_weights.shape[1])\n        pool_focus = pool_weights.max(dim=1).values\n        self_entropy, self_focus = self._attention_summary(self_attention_maps)\n\n        return {\n            "logits": logits,\n            "attention_weights": pool_weights,\n            "attention_entropy": pool_entropy,\n            "attention_focus": pool_focus,\n            "self_attention_entropy": self_entropy,\n            "self_attention_focus": self_focus,\n            "semantic_feature": pooled,\n        }\n\nclass EfficientNetBinaryClassifier(nn.Module):\n    def __init__(self):\n        super().__init__()\n        import timm\n        self.model = timm.create_model("efficientnet_b0", pretrained=False, num_classes=1, drop_rate=0.2)\n\n    def forward(self, x):\n        return self.model(x).squeeze(1)\n\n\nclass ViTSemanticAttentionClassifier(nn.Module):\n    def __init__(self, backbone):\n        super().__init__()\n        self.vit = backbone\n        self.semantic_head = SemanticTransformerHead(hidden_size=backbone.config.hidden_size)\n        self.vit.requires_grad_(False)\n\n    def forward(self, pixel_values):\n        # The saved audit used cuDNN\'s default TF32 convolution arithmetic.\n        # Scope it to ViT so CF and the other branches retain their own settings.\n        precision = torch.backends.cudnn.flags(\n            enabled=torch.backends.cudnn.enabled,\n            benchmark=torch.backends.cudnn.benchmark,\n            deterministic=torch.backends.cudnn.deterministic,\n            allow_tf32=True,\n        ) if pixel_values.device.type == "cuda" else nullcontext()\n        with precision:\n            output = self.vit(pixel_values=pixel_values, return_dict=True)\n        return self.semantic_head(output.last_hidden_state[:, 1:, :])\n\n\ndef load_original_models(config):\n    from transformers import AutoImageProcessor, AutoModel\n    # Original files are user\'s own training artifacts; preflight verifies their hashes.\n    effnet = EfficientNetBinaryClassifier()\n    saved = torch.load(config.checkpoints["effnet"], map_location="cpu", weights_only=False)\n    state = saved["model_state"]\n    if not all(k.startswith("model.") for k in state):\n        state = {"model." + k: v for k, v in state.items()}\n    effnet.load_state_dict(state, strict=True)\n\n    processor = AutoImageProcessor.from_pretrained(SEMANTIC_ID, revision=SEMANTIC_REVISION, use_fast=True)\n    backbone = AutoModel.from_pretrained(SEMANTIC_ID, revision=SEMANTIC_REVISION)\n    semantic = ViTSemanticAttentionClassifier(backbone)\n    saved = torch.load(config.checkpoints["semantic"], map_location="cpu", weights_only=False)\n    if saved.get("config", {}).get("freeze_vit_backbone", True) is not True:\n        raise ValueError("Cannot restore only the head of a trained backbone")\n    state = {k.removeprefix("semantic_head."): v for k, v in saved["model_state"].items()\n             if k.startswith("semantic_head.")}\n    if not state:\n        raise ValueError("Semantic head weights are missing")\n    semantic.semantic_head.load_state_dict(state, strict=True)\n    return {"effnet": effnet.eval(), "semantic": semantic.eval()}, processor\n\n', 'src/ai_shield_metrics.py': '"""Shared metrics for AI Shield binary detectors and fusion models."""\n\nfrom __future__ import annotations\n\nimport numpy as np\nfrom sklearn.metrics import (\n    accuracy_score,\n    average_precision_score,\n    brier_score_loss,\n    confusion_matrix,\n    f1_score,\n    precision_score,\n    recall_score,\n    roc_auc_score,\n)\n\n\ndef binary_metrics(y_true, y_prob, threshold: float = 0.5) -> dict[str, float | int]:\n    """Compute the binary metrics used across the AI Shield notebooks."""\n    y_true_array = np.asarray(y_true).astype(int)\n    y_prob_array = np.asarray(y_prob).astype(float)\n    y_pred = (y_prob_array >= threshold).astype(int)\n    tn, fp, fn, tp = confusion_matrix(y_true_array, y_pred, labels=[0, 1]).ravel()\n    return {\n        "auc": float(roc_auc_score(y_true_array, y_prob_array)),\n        "average_precision": float(average_precision_score(y_true_array, y_prob_array)),\n        "accuracy": float(accuracy_score(y_true_array, y_pred)),\n        "precision": float(precision_score(y_true_array, y_pred, zero_division=0)),\n        "recall": float(recall_score(y_true_array, y_pred, zero_division=0)),\n        "f1": float(f1_score(y_true_array, y_pred, zero_division=0)),\n        "brier": float(brier_score_loss(y_true_array, y_prob_array)),\n        "threshold": float(threshold),\n        "n": int(len(y_true_array)),\n        "tn": int(tn),\n        "fp": int(fp),\n        "fn": int(fn),\n        "tp": int(tp),\n    }\n\n\ndef threshold_sweep(y_true, y_prob, thresholds=None, metric: str = "f1") -> tuple[float, list[dict[str, float | int]]]:\n    """Evaluate metrics across thresholds and return the best threshold."""\n    if thresholds is None:\n        thresholds = np.linspace(0.01, 0.99, 99)\n    rows = [binary_metrics(y_true, y_prob, float(threshold)) for threshold in thresholds]\n    best = max(rows, key=lambda row: float(row[metric]))\n    return float(best["threshold"]), rows\n', 'src/ai_shield_cf_inference.py': '"""Frozen Community Forensics inference, matching finetune_v1 evaluation."""\nimport hashlib\nfrom pathlib import Path\n\nimport torch\nfrom torch import nn\nimport timm\nfrom PIL import Image, ImageOps\nfrom torchvision import transforms\n\n\ndef file_sha256(path):\n    digest = hashlib.sha256()\n    with Path(path).open("rb") as handle:\n        for chunk in iter(lambda: handle.read(1024 * 1024), b""):\n            digest.update(chunk)\n    return digest.hexdigest()\n\n\nclass CommunityForensicsDetector(nn.Module):\n    def __init__(self):\n        super().__init__()\n        self.vit = timm.create_model(\n            "vit_small_patch16_384.augreg_in21k_ft_in1k", pretrained=False,\n        )\n        self.vit.head = nn.Linear(self.vit.num_features, 1)\n\n    def forward(self, images):\n        return self.vit(images)\n\n\nclass FrozenCF:\n    def __init__(self, checkpoint_path, expected_sha256, threshold,\n                 expected_manifest_sha256, device=None):\n        if file_sha256(checkpoint_path) != expected_sha256:\n            raise RuntimeError("Checkpoint SHA-256 mismatch.")\n        if not 0 <= threshold <= 1:\n            raise ValueError("Threshold must be in [0, 1].")\n        self.threshold = float(threshold)\n        self.device = torch.device(device or ("cuda" if torch.cuda.is_available() else "cpu"))\n        if self.device.type == "cuda":\n            torch.backends.cuda.matmul.allow_tf32 = False\n            torch.backends.cudnn.allow_tf32 = False\n        self.model = CommunityForensicsDetector()\n        payload = torch.load(checkpoint_path, map_location="cpu", weights_only=True)\n        if payload.get("stage") != "stage2_last_blocks":\n            raise RuntimeError("Checkpoint is not Stage 2.")\n        if payload.get("experiment_manifest_sha256") != expected_manifest_sha256:\n            raise RuntimeError("Checkpoint belongs to a different experiment manifest.")\n        self.best_val_auc = float(payload["best_val_auc"])\n        self.model.load_state_dict(payload["model_state"], strict=True)\n        self.model.requires_grad_(False)\n        self.model.float().eval().to(self.device)\n        self.preprocess = transforms.Compose([\n            transforms.Resize(440, interpolation=transforms.InterpolationMode.BILINEAR,\n                              antialias=True),\n            transforms.CenterCrop(384),\n            transforms.ToTensor(),\n            transforms.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),\n        ])\n\n    def predict(self, path):\n        with Image.open(path) as source:\n            source.seek(0)\n            image = ImageOps.exif_transpose(source).convert("RGB")\n            tensor = self.preprocess(image).unsqueeze(0).to(self.device)\n        with torch.inference_mode():\n            logits = self.model(tensor).reshape(-1)\n            if logits.numel() != 1 or not torch.isfinite(logits).all():\n                raise RuntimeError("Invalid model output.")\n            score = float(torch.sigmoid(logits)[0].cpu())\n            logit = float(logits[0].cpu())\n        return {"logit": logit, "prob_fake": score,\n                "predicted_label": int(score >= self.threshold)}\n', 'src/__init__.py': ''}
EMBEDDED_HASHES = {'src/ai_shield_adaptation.py': 'd10eb8cb82247a413d3d27c7bd8a1ababbe9129858bbae3164adfa79c39f4410', 'src/ai_shield_adaptation_models.py': '4651eed117e378d5afd01c2715861ce8f66c0e64346f5a6ece4a32aa94e47e1d', 'src/ai_shield_metrics.py': 'd4f40c0ee1a152f8e6453e5206696e9ebc19107b91f9a354ad7d1008eb8de2d2', 'src/ai_shield_cf_inference.py': '12b03eeade84fe7027ad70cc2782fb85dd857ec2e681e764b8b936f9b3725c52', 'src/__init__.py': 'e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855'}

import hashlib
from pathlib import Path
CODE_ROOT = Path('/content/ai_shield_adaptation_code')
for relative, source in EMBEDDED_SOURCES.items():
    assert hashlib.sha256(source.encode()).hexdigest() == EMBEDDED_HASHES[relative]
    path = CODE_ROOT / relative
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(source, encoding='utf-8')
sys.path.insert(0, str(CODE_ROOT))
print('Research modules loaded. Original project files are untouched.')


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import json, gc, platform
import pandas as pd
import torch
from IPython.display import display
from src.ai_shield_adaptation import *
from src.ai_shield_adaptation_models import load_original_models

CONFIG = Config()
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
seed_everything(CONFIG.seed)
print('Device:', DEVICE)
print('GPU:', torch.cuda.get_device_name(0) if DEVICE == 'cuda' else 'No GPU attached')
print('Training budget:', CONFIG.head_epochs, '+', CONFIG.partial_epochs, 'epochs per branch')


## 2. Verify inputs and reserve fusion-training images
Reads the same Drive paths recorded by the completed CF and saved-baseline notebooks.
Verifies split hashes, image bytes, labels, duplicates, group boundaries and original checkpoints.
Creates a new timestamped folder under `MyDrive/AI_Shield_Adaptation/two_branch_v1/`.
Run this cell only once for an experiment; starting it again creates a separate run.


In [ ]:
frames, audit = preflight(CONFIG)
RUN, partition = prepare_run(CONFIG, frames)
write_json(RUN / 'embedded_source_hashes.json', EMBEDDED_HASHES)
display(partition.groupby(['role', 'label']).size().rename('images').reset_index())
print('Output:', RUN)


## 3. Restore the original networks and check implementation parity
Preserves the audited old-model preprocessing (no EXIF transpose), not CF's preprocessing.
Loads the semantic head strictly and the pinned, originally frozen ViT backbone separately.
Six fixed historical test images are used only for reproduction checking, never training or selection.


In [ ]:
models, processor = load_original_models(CONFIG)
parity = check_audit_parity(models, processor, frames, CONFIG, DEVICE)
write_json(RUN / 'baseline_parity.json', parity)
print('Baseline parity PASS on', parity['n'], 'fixed images.')


## 4. Experiment A: frozen networks, newly fitted two-branch fusion
Fits a standardized logistic regression on the reserved fusion-training images.
Selects each score's max-F1 threshold on validation only. The old calibration is not reused.
These are bounded detector scores, not established calibrated probabilities.


In [ ]:
frozen_fusion, frozen_thresholds = fit_variant(
    'frozen_pair', models, processor, partition, frames['validation'], CONFIG, RUN, DEVICE)
print('Frozen-pair validation thresholds:', frozen_thresholds)


## 5. Experiment B: limited adaptation, then refit fusion
Each branch starts from its original saved weights. Stage 1 trains the classifier/semantic head.
Stage 2 also trains the final EfficientNet block + output convolution/normalization, or the final ViT block + normalization.
Frozen BatchNorm statistics stay frozen. Checkpoints are chosen by validation AUROC, including the original as a candidate.
Fusion images are excluded from branch training. No hyperparameter search is performed.


In [ ]:
if DEVICE != 'cuda':
    raise RuntimeError('Select a GPU runtime before full training. CPU unit tests are separate.')
training_reports = []
branch_train = partition[partition.role == 'branch_train'].copy()
for branch, model in models.items():
    training_reports.append(train_branch(model, branch, processor, branch_train,
        frames['validation'], CONFIG, RUN, DEVICE))
adapted_fusion, adapted_thresholds = fit_variant(
    'adapted_pair', models, processor, partition, frames['validation'], CONFIG, RUN, DEVICE)
display(pd.DataFrame(training_reports))


## 6. Lock selections before final comparisons
No choosing thresholds, epochs or fusion weights from test results.
A lock prevents accidentally mixing checkpoints from different cell runs.


In [ ]:
selected_files = ['effnet_adapted.pt', 'semantic_adapted.pt', 'frozen_pair_fusion.joblib',
    'adapted_pair_fusion.joblib', 'frozen_pair_selection.json', 'adapted_pair_selection.json']
selection_lock = {name: sha256_file(RUN / name) for name in selected_files}
write_json(RUN / 'selection_lock.json', selection_lock)
print('Selections locked. Historical evaluation can now run.')


In [ ]:
locked = json.loads((RUN / 'selection_lock.json').read_text())
assert all(sha256_file(RUN / name) == value for name, value in locked.items())
comparison_rows = evaluate_variant('adapted_pair', models, processor, frames,
    adapted_fusion, adapted_thresholds, CONFIG, RUN, DEVICE)
original_models, original_processor = load_original_models(CONFIG)
comparison_rows += evaluate_variant('frozen_pair', original_models, original_processor, frames,
    frozen_fusion, frozen_thresholds, CONFIG, RUN, DEVICE)
del original_models, original_processor
gc.collect()
torch.cuda.empty_cache()
comparison_rows += historical_references(frames, CONFIG, audit, RUN)
comparison = pd.DataFrame(comparison_rows)
comparison.to_csv(RUN / 'comparison.csv', index=False)
display(comparison[['model', 'split', 'auc', 'f1', 'accuracy', 'fpr', 'threshold']])


## 7. Same-runtime timing: adapted pair versus frozen CF
Uses 64 fixed validation images (32 per class), three warmups and three timed passes, batch size 1.
Includes file reading, preprocessing, inference and score combination. Excludes loading weights, API and provenance work.
Process RSS is not isolated peak model memory. These timings are not a public-deployment throughput guarantee.


In [ ]:
timing_images = frames['validation'].sort_values('sample_id').groupby('label', group_keys=False).head(32)
paths = timing_images.local_path.tolist()
timing = {'device': DEVICE, 'gpu': torch.cuda.get_device_name(0) if DEVICE == 'cuda' else None,
          'platform': platform.platform(), 'torch': torch.__version__, 'cpu_threads': torch.get_num_threads(),
          'sample_ids': timing_images.sample_id.tolist()}
pair = PairPredictor(models, processor, adapted_fusion, DEVICE)
timing['adapted_pair'] = benchmark(pair.predict, paths, DEVICE)
for model in models.values():
    model.cpu()
del pair
gc.collect()
torch.cuda.empty_cache()

from src.ai_shield_cf_inference import FrozenCF
cf = FrozenCF(CONFIG.cf_root / 'checkpoints/stage2_last_blocks/best.pt',
    expected_sha256='51532d28870706a3c380a29889f634c556583bd62ed74775e5e610d5fd9d7b5f',
    threshold=0.6339887976646423,
    expected_manifest_sha256='2ba189a3be70e6840b6be6b530253925e87d7782c2882f0189ce8e582f20d01d',
    device=DEVICE)
cf_live = pd.DataFrame([{'sample_id': row.sample_id, 'label': row.label,
    **cf.predict(row.local_path)} for row in timing_images.itertuples()])
cf_saved = checked_csv(CONFIG.cf_root / 'predictions/finetuned_validation.csv')
cf_saved = cf_saved[cf_saved.sample_id.isin(timing_images.sample_id)]
cf_saved = align_predictions(timing_images, cf_saved, ['prob_fake'])
assert np.allclose(cf_live.prob_fake, cf_saved.prob_fake, atol=1e-5, rtol=1e-5), 'CF runtime parity failed'
timing['fine_tuned_cf'] = benchmark(cf.predict, paths, DEVICE)
write_json(RUN / 'runtime_comparison.json', timing)
del cf
torch.cuda.empty_cache()
display(pd.DataFrame([{'model': k, **timing[k]} for k in ('adapted_pair', 'fine_tuned_cf')]))


## 8. Final report and interpretation
Compare frozen vs adapted networks, each branch vs the pair, and the pair vs CF.
Check F1 AND real-image false alarms, not just AUROC. Small score differences need uncertainty analysis/repeat runs.
Record a negative result honestly. A useful result may be cheaper adaptation without surpassing CF.
No result automatically replaces the app detector. JPEG/resize robustness, non-photographic real images,
matched real/generated content and a fresh confirmation set remain separate evaluation tasks.


In [ ]:
report = {
    'status': 'completed', 'run': str(RUN), 'scope': 'two_branch_variant_not_original_full_pipeline',
    'baseline_parity': parity, 'training': training_reports, 'comparison': comparison_rows,
    'runtime': timing, 'selection_lock': selection_lock,
    'caveats': ['Historical tests were previously inspected; fresh confirmation is still needed.',
        'Branches saw fewer images than CF because fusion training is reserved inside the 2990-image budget.',
        'No claim of matched pretraining cost, isolated memory peak or calibrated probabilities.',
        'Generated-vs-real source/content confounding is not resolved by this experiment.',
        'This is not a full 17-feature pipeline retraining; VAE and face branches are excluded.'],
}
write_json(RUN / 'final_report.json', report)
print('Saved:', RUN / 'final_report.json')
print('Frozen CF application remains unchanged.')


## Sources and implementation notes
- Original model definitions: local `notebooks/phase2_semantic_report.ipynb` and the audited saved-baseline notebook.
- [Saved baseline audit](https://colab.research.google.com/drive/10jl7g0qknxLUCGF1VddbqbVf43oef4oD)
- [Completed CF experiment](https://colab.research.google.com/drive/1q5CB7k9fdY9ylfqMDQgClVl4k_ogNJef)
- [Stacking and overfitting](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.StackingClassifier.html)
- [Gradient modes and evaluation mode](https://docs.pytorch.org/docs/stable/notes/autograd.html)
